## Plot Abil Output

In [ ]:
import cartopy.crs as ccrs
import cartopy as cart
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import xarray as xr
import sys
from pathlib import Path
from yaml import load, Loader
from matplotlib import cm
from matplotlib.colors import LogNorm
import matplotlib.ticker as ticker

In [ ]:
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in [start, *start.parents]:
        if (path / "environment.yml").exists():
            return path
    raise FileNotFoundError("Could not find project root containing environment.yml")


PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "2-phase example" / "2-phase.yml"

with CONFIG_PATH.open('r') as f:
    model_config = load(f, Loader=Loader)

run_name = 'ghux_so'
file = '2026-05-04_mean_abundance'
post_dir = PROJECT_ROOT / model_config['path_out'] / run_name / 'posts'
ds = xr.open_dataset(post_dir / f"{file}.nc")


In [ ]:
print(ds)

### Create of map of surface Gephyrocapsa huxleyi HET abundance in April

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = plt.axes(projection=ccrs.SouthPolarStereo())

# Add coastlines and land fill color
ax.coastlines()
ax.add_feature(cart.feature.LAND, 
               linewidth = 1, 
               zorder=3, 
               edgecolor='k', 
               facecolor="lightgray")

ax.gridlines(draw_labels=True, 
             linewidth=1,
             color='gray',
             alpha=0.8,
             linestyle='-',
             zorder=1
)
p = ds['Gephyrocapsa huxleyi HET'].sel(
    depth=0, 
    time=1
    ).plot(
        x='lon',
        y='lat',
        transform=ccrs.PlateCarree(),
        add_colorbar=False,)


cbar = plt.colorbar(p, 
                    location='bottom', 
                    label="Gephyrocapsa huxleyi HET (cells L$^{-1}$)")

### Create map of depth integrated PIC stock for Gephyrocapsa huxleyi HET

In [ ]:
file = '2026-05-04_mean_pic'
ds = xr.open_dataset(post_dir / f"{file}.nc")

fig = plt.figure(figsize=(8, 6))
ax = plt.axes(projection=ccrs.SouthPolarStereo())

# Add coastlines and land fill color
ax.coastlines()
ax.add_feature(cart.feature.LAND, 
               linewidth = 1, 
               zorder=3, 
               edgecolor='k', 
               facecolor="lightgray")

ax.gridlines(draw_labels=True, 
             linewidth=1,
             color='gray',
             alpha=0.8,
             linestyle='-',
             zorder=1
)

conversion_factor = 1e3 * 1e-9 # pg C L-1 to pg C m-3, pg C m-3 to mg C m-3
depth_width = 5
depth_integrated_pp = ds['Gephyrocapsa huxleyi HET'].sum(dim=["depth"]).mean(dim="time") * depth_width * conversion_factor

p = depth_integrated_pp.plot(
        x='lon',
        y='lat',
        transform=ccrs.PlateCarree(),
        add_colorbar=False,)


cbar = plt.colorbar(p, 
                    location='bottom', 
                    label="Gephyrocapsa huxleyi HET PIC (mg m$^{-2}$ d$^{-1}$)")
